In [5]:
from tensorflow.keras.models import load_model
import numpy as np
import pandas as pd

## Load model

In [3]:
# Charger le modèle
model = load_model("../models/lstm_model(50).h5")

# Vérifier l’architecture
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 50)             │        10,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │           102 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,504 (41.04 KB)

 Trainable params: 10,502 (41.02 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

# Load dataset

In [7]:

df = pd.read_csv("../data/dataset_lstm.csv")
df.head()

,Unnamed: 0,R_-239,R_-238,R_-237,R_-236,R_-235,R_-234,R_-233,R_-232,R_-231,...,R_-6,R_-5,R_-4,R_-3,R_-2,R_-1,R_0,ticker,date,target
0,0,-0.768984,0.008414,0.181141,-0.289120,-3.684832,1.237274,-1.277278,-1.034766,0.263101,...,1.106459,0.515260,0.162772,0.220213,0.138572,0.069141,-1.701827,A,2014-01-23,0
1,1,0.008414,0.181141,-0.289120,-3.684832,1.237274,-1.277278,-1.034766,0.263101,-0.877796,...,0.515260,0.162772,0.220213,0.138572,0.069141,-1.701827,-1.907957,A,2014-01-24,1
2,2,0.181141,-0.289120,-3.684832,1.237274,-1.277278,-1.034766,0.263101,-0.877796,-0.565976,...,0.162772,0.220213,0.138572,0.069141,-1.701827,-1.907957,0.497611,A,2014-01-27,0
3,3,-0.289120,-3.684832,1.237274,-1.277278,-1.034766,0.263101,-0.877796,-0.565976,1.276668,...,0.220213,0.138572,0.069141,-1.701827,-1.907957,0.497611,-0.263311,A,2014-01-28,1
4,4,-3.684832,1.237274,-1.277278,-1.034766,0.263101,-0.877796,-0.565976,1.276668,-0.442705,...,0.138572,0.069141,-1.701827,-1.907957,0.497611,-0.263311,-0.565664,A,2014-01-29,1


In [17]:
def normalize_and_split_study_period(df, train_days=750):
    df_seq = df
    
    # 1. Encontrar los días únicos en este bloque exacto
    unique_dates = sorted(df_seq['date'].unique())
    
    # Si por alguna razón el bloque tiene menos de 750 días, ajustamos el índice
    split_idx = min(train_days - 1, len(unique_dates) - 1)
    split_date = unique_dates[split_idx] 
    
    # 2. Separar Train y Test estrictamente por cronología
    train_df = df_seq[df_seq['date'] <= split_date].copy()
    test_df = df_seq[df_seq['date'] > split_date].copy()
    
    # 3. Obtener solo las columnas matemáticas (R_-239 a R_0)
    return_cols = [col for col in df_seq.columns if col.startswith('R_')]

    return train_df[return_cols+["target"]], test_df[return_cols+["target"]]
    
return_cols = [col for col in df.columns if col.startswith('R_')]

train_df, test_df = normalize_and_split_study_period(df)


X_train = train_df[return_cols]
y_train = train_df["target"]

X_test = test_df[return_cols]
y_test = test_df["target"]

# Accuracy

In [18]:
loss, accuracy = model.evaluate(X_train, y_train)

print("Loss:", loss)
print("Accuracy:", accuracy)



10676/10676 ━━━━━━━━━━━━━━━━━━━━ 97s 9ms/step - accuracy: 0.5307 - loss: 0.6889
Loss: 0.6889145374298096
Accuracy: 0.5306650996208191


In [19]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Loss:", loss)
print("Accuracy:", accuracy)

3060/3060 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - accuracy: 0.5100 - loss: 0.6938
Loss: 0.6938315629959106
Accuracy: 0.510049045085907


In [ ]:
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Make predictions on test set
y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()
y_test_binary = y_test.values.astype(int)

# ===== CLASSIFICATION METRICS =====
print("=" * 50)
print("CLASSIFICATION METRICS")
print("=" * 50)

# Confusion Matrix
cm = confusion_matrix(y_test_binary, y_pred)
print("\nConfusion Matrix:")
print(cm)

# F1-Score
f1 = f1_score(y_test_binary, y_pred)
print(f"\nF1-Score: {f1:.4f}")

# Global Accuracy
accuracy = accuracy_score(y_test_binary, y_pred)
print(f"Global Accuracy: {accuracy:.4f}")

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test_binary, y_pred))

# Visualize Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True)
plt.title('Confusion Matrix - Test Set')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# ===== FINANCIAL METRICS =====
print("\n" + "=" * 50)
print("FINANCIAL METRICS")
print("=" * 50)

# Daily returns from test set
daily_returns = X_test[return_cols].values
# Get the last return column (R_0 represents current day return)
actual_returns = X_test['R_0'].values if 'R_0' in X_test.columns else daily_returns[:, -1]

# Strategy returns based on predictions (1 = go long, 0 = no position or short)
strategy_returns = actual_returns * y_pred

# 1. Cumulative Return
cumulative_return = np.sum(strategy_returns)
cumulative_return_pct = (np.exp(np.sum(np.log(1 + strategy_returns + 1e-8))) - 1) * 100
print(f"\nCumulative Return: {cumulative_return:.4f}")
print(f"Cumulative Return (%): {cumulative_return_pct:.2f}%")

# 2. Volatility (annualized)
daily_volatility = np.std(strategy_returns)
annualized_volatility = daily_volatility * np.sqrt(252)  # 252 trading days per year
print(f"\nDaily Volatility: {daily_volatility:.4f}")
print(f"Annualized Volatility: {annualized_volatility:.4f}")

# 3. Sharpe Ratio (assuming risk-free rate = 0)
risk_free_rate = 0
mean_return = np.mean(strategy_returns)
sharpe_ratio = (mean_return - risk_free_rate) / (daily_volatility + 1e-8) * np.sqrt(252)
print(f"\nMean Daily Return: {mean_return:.4f}")
print(f"Sharpe Ratio (annualized): {sharpe_ratio:.4f}")

# Benchmark comparison (buy and hold)
benchmark_cumulative = np.sum(actual_returns)
print(f"\nBenchmark (Buy & Hold) Cumulative Return: {benchmark_cumulative:.4f}")
print(f"Strategy vs Benchmark: {cumulative_return - benchmark_cumulative:.4f}")
